In [0]:

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

# The exact MLflow model URI verified earlier.
model_uri = (
    "models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d"
)

model = mlflow.sklearn.load_model(model_uri)

threshold = 0.55

feature_cols = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Month",
    "CRSDepTime",
    "DepHour",
    "DayOfWeek",
    "IsWeekend",
    "Quarter"
]

print("Model URI:", model_uri)
print("Decision threshold:", threshold)
print("Required inputs:", feature_cols)

print("PASS: Original fitted model loaded")
print("PASS: Deployment inputs defined")


Model URI: models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d
Decision threshold: 0.55
Required inputs: ['Reporting_Airline', 'Origin', 'Dest', 'Month', 'CRSDepTime', 'DepHour', 'DayOfWeek', 'IsWeekend', 'Quarter']
PASS: Original fitted model loaded
PASS: Deployment inputs defined


In [0]:

# Example input for a scheduled flight.
# The model accepts PRECOMPUTED features.

flight_input = pd.DataFrame([{
    "Reporting_Airline": "AA",
    "Origin": "JFK",
    "Dest": "LAX",
    "Month": 12,
    "CRSDepTime": 1800,
    "DepHour": 18,
    "DayOfWeek": 2,
    "IsWeekend": 0,
    "Quarter": 4
}])

delay_probability = float(
    model.predict_proba(flight_input[feature_cols])[0, 1]
)

predicted_class = int(
    delay_probability >= threshold
)

print("FLIGHT DELAY PREDICTION")
print("Airline:", flight_input.iloc[0]["Reporting_Airline"])
print("Route: JFK to LAX")
print("Delay probability:", round(delay_probability, 4))
print("Prediction:", predicted_class)

if predicted_class == 1:
    print("Interpretation: Predicted delay of 15+ minutes")
else:
    print("Interpretation: Predicted on-time arrival")

print("PASS: Valid flight prediction completed")


FLIGHT DELAY PREDICTION
Airline: AA
Route: JFK to LAX
Delay probability: 0.5914
Prediction: 1
Interpretation: Predicted delay of 15+ minutes
PASS: Valid flight prediction completed


In [0]:

# Input validation function for the planned deployment.

required_columns = feature_cols

def validate_flight_input(record):
    if not isinstance(record, dict):
        raise ValueError("Input must be a JSON object")

    missing = [
        column for column in required_columns
        if column not in record
    ]

    if missing:
        raise ValueError(
            f"Missing required fields: {missing}"
        )

    if any(
        pd.isna(record[column])
        for column in required_columns
    ):
        raise ValueError("Input contains null values")

    for column in [
        "Reporting_Airline", "Origin", "Dest"
    ]:
        if not isinstance(record[column], str) or not record[column].strip():
            raise ValueError(f"Invalid {column}")

    for column in [
        "Month", "CRSDepTime", "DepHour",
        "DayOfWeek", "IsWeekend", "Quarter"
    ]:
        value = record[column]
        if isinstance(value, (bool, np.bool_)) or not isinstance(
            value, (int, np.integer)
        ):
            raise ValueError(f"{column} must be an integer")

    month = record["Month"]
    scheduled = record["CRSDepTime"]
    expected_hour = (
        0 if scheduled == 2400 else scheduled // 100
    )

    if not 1 <= month <= 12:
        raise ValueError("Month must be 1-12")

    if not (
        scheduled == 2400 or
        (0 <= scheduled <= 2359 and scheduled % 100 < 60)
    ):
        raise ValueError("Invalid scheduled departure time")

    if record["DepHour"] != expected_hour:
        raise ValueError("DepHour does not match CRSDepTime")

    if record["Quarter"] != (month - 1) // 3 + 1:
        raise ValueError("Quarter does not match Month")

    if not 1 <= record["DayOfWeek"] <= 7:
        raise ValueError("DayOfWeek must be 1-7")

    expected_weekend = int(record["DayOfWeek"] in (1, 7))
    if record["IsWeekend"] != expected_weekend:
        raise ValueError("IsWeekend does not match DayOfWeek")

    return True

# Successful scenario
valid_record = flight_input.iloc[0].to_dict()
assert validate_flight_input(valid_record)

print("PASS: Valid input accepted")

# Controlled invalid-input scenario
bad_record = dict(valid_record)
bad_record.pop("Origin")

failure_detected = False

try:
    validate_flight_input(bad_record)
except ValueError as error:
    failure_detected = True
    print("EXPECTED INVALID INPUT:", str(error))

assert failure_detected

print("PASS: Invalid input rejected correctly")


PASS: Valid input accepted
EXPECTED INVALID INPUT: Missing required fields: ['Origin']
PASS: Invalid input rejected correctly
